# StreamVault Data Profile

Investigation of the supplied movie dataset, its data quality and the records used in the licensing analysis.

In [12]:
from pathlib import Path
import pandas as pd

# Project paths
HERE = Path.cwd()

PROJECT_ROOT = next(
    (p for p in [HERE, *HERE.parents]
     if (p / "data" / "processed").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("StreamVault project directory not found.")

DATA = PROJECT_ROOT / "data"

# Load datasets
movies = pd.read_csv(DATA / "processed" / "analysis_movies.csv")
credits = pd.read_csv(DATA / "processed" / "credits.csv")
audit = pd.read_csv(DATA / "processed" / "id_audit.csv")
quarantine = pd.read_csv(DATA / "quarantine" / "quarantined_records.csv")

print("Movies:", movies.shape)
print("Credits:", credits.shape)
print("ID audit:", audit.shape)
print("Quarantine:", quarantine.shape)

Movies: (112, 22)
Credits: (32759, 8)
ID audit: (126, 5)
Quarantine: (14, 5)


## 1. The data, as prepared

In [13]:
movies.head()

,id,title,original_title,release_date,runtime,genres,budget,revenue,popularity,vote_average,vote_count,original_language,overview,status,belongs_to_collection_id,belongs_to_collection_name,production_companies,production_countries,release_year,candidate_type,gross_minus_budget,revenue_to_budget
0,155,The Dark Knight,The Dark Knight,2008-07-16,152,Action; Thriller; Crime,185000000,1008480529,52.5171,8.500,37425,en,Batman raises the stakes in his war on crime. ...,Released,263.0,The Dark Knight Collection,Warner Bros. Pictures; Legendary Pictures; Syn...,United Kingdom; United States of America,2008,Franchise,823480529,5.451246
1,165,Back to the Future Part II,Back to the Future Part II,1989-11-22,108,Adventure; Comedy; Science Fiction,40000000,332861197,9.8404,7.784,14537,en,Marty and Doc are at it again as the time-trav...,Released,264.0,Back to the Future Collection,Universal Pictures; Amblin Entertainment,United States of America,1989,Franchise,292861197,8.321530
2,242,The Godfather Part III,The Godfather Part III,1990-12-25,162,Crime; Drama; Thriller,54000000,136766062,13.6773,7.413,7204,en,In the midst of trying to legitimize his busin...,Released,230.0,The Godfather Collection,Paramount Pictures; American Zoetrope,United States of America,1990,Franchise,82766062,2.532705
3,49026,The Dark Knight Rises,The Dark Knight Rises,2012-07-17,165,Action; Crime; Thriller,250000000,1081041287,20.6789,7.801,25343,en,Following the death of District Attorney Harve...,Released,263.0,The Dark Knight Collection,Syncopy; Legendary Pictures; Warner Bros. Pict...,United Kingdom; United States of America,2012,Franchise,831041287,4.324165
4,8077,Alien³,Alien³,1992-05-22,114,Science Fiction; Action; Horror,50000000,159773545,7.9407,6.391,6727,en,After escaping with Newt and Hicks from the al...,Released,8091.0,Alien Collection,20th Century Fox; Brandywine Productions,United States of America,1992,Franchise,109773545,3.195471


## 2. Every column: missing values, distinct values, commonest values

In [14]:
def profile_columns(df):
    return pd.DataFrame({
        "missing": df.isna().sum(),
        "distinct": df.nunique(),
        "commonest values": [
            ", ".join(
                df[col].dropna().astype(str)
                .value_counts().head(3).index
            )
            for col in df.columns
        ]
    })

profile_columns(movies)

,missing,distinct,commonest values
id,0,112,"155, 165, 242"
title,0,112,"The Dark Knight, Back to the Future Part II, T..."
original_title,0,112,"The Dark Knight, Back to the Future Part II, T..."
release_date,0,112,"2008-07-16, 1989-11-22, 1990-12-25"
runtime,0,64,"92, 124, 136"
genres,0,66,"Adventure; Action; Science Fiction, Adventure;..."
budget,0,60,"150000000, 250000000, 200000000"
revenue,0,112,"1008480529, 332861197, 136766062"
popularity,0,112,"52.5171, 9.8404, 13.6773"
vote_average,0,108,"8.5, 7.97, 7.818"


## 3. The data quality checks

In [15]:
def checks(movies, audit):
    return {
        "movie records": len(movies),
        "duplicate movie IDs": int(movies["id"].duplicated().sum()),
        "duplicate rows": int(movies.duplicated().sum()),
        "missing budgets": int(movies["budget"].isna().sum()),
        "zero budgets": int((movies["budget"] == 0).sum()),
        "negative budgets": int((movies["budget"] < 0).sum()),
        "missing revenues": int(movies["revenue"].isna().sum()),
        "zero revenues": int((movies["revenue"] == 0).sum()),
        "missing runtimes": int(movies["runtime"].isna().sum()),
        "zero runtimes": int((movies["runtime"] == 0).sum()),
        "missing collection IDs": int(
            movies["belongs_to_collection_id"].isna().sum()
        ),
        "standalone films": int(
            (movies["candidate_type"] == "Standalone").sum()
        ),
        "franchise films": int(
            (movies["candidate_type"] == "Franchise").sum()
        ),
        "earliest release year": int(movies["release_year"].min()),
        "latest release year": int(movies["release_year"].max()),
        "films from 2000 onward": int(
            (movies["release_year"] >= 2000).sum()
        ),
        "audit records": len(audit),
        "quarantined records": len(quarantine)
    }

pd.Series(checks(movies, audit), name="Result").to_frame()

,Result
movie records,112
duplicate movie IDs,0
duplicate rows,0
missing budgets,0
zero budgets,0
negative budgets,0
missing revenues,0
zero revenues,0
missing runtimes,0
zero runtimes,0


## 4. The rows behind the numbers

Counts show how many records are affected. These records show what was identified.

In [16]:
# Identifier audit: records that were not successfully retrieved
audit[audit["record_status"] != "success"]

,source_row,tmdb_id,normalized_tmdb_id,record_status,canonical_tmdb_id
16,18,8471293,8471293,both_failed,8471293
28,30,-1,-1,invalid_id,-1
33,35,9412887,9412887,both_failed,9412887
41,43,603,603,duplicate_id,603
58,60,278,278,duplicate_id,278
70,72,8471294,8471294,both_failed,8471294
87,89,1891,1891,duplicate_id,1891
88,90,155,155,duplicate_id,155
90,92,9412888,9412888,both_failed,9412888
91,93,550,550,duplicate_id,550


In [17]:
# Films without an associated TMDb collection
movies[
    movies["belongs_to_collection_id"].isna()
][["id", "title", "release_year", "candidate_type"]]

,id,title,release_year,candidate_type
6,680,Pulp Fiction,1994,Standalone
14,769,GoodFellas,1990,Standalone
24,278,The Shawshank Redemption,1994,Standalone
36,1422,The Departed,2006,Standalone
37,27205,Inception,2010,Standalone
50,1124,The Prestige,2006,Standalone
52,111,Scarface,1983,Standalone
54,10681,WALL·E,2008,Standalone
55,641,Requiem for a Dream,2000,Standalone
63,129,Spirited Away,2001,Standalone


In [18]:
# Quarantined records
quarantine.head(14)

,source_row,tmdb_id,normalized_tmdb_id,record_status,canonical_tmdb_id
0,18,8471293,8471293,both_failed,8471293
1,30,-1,-1,invalid_id,-1
2,35,9412887,9412887,both_failed,9412887
3,43,603,603,duplicate_id,603
4,60,278,278,duplicate_id,278
5,72,8471294,8471294,both_failed,8471294
6,89,1891,1891,duplicate_id,1891
7,90,155,155,duplicate_id,155
8,92,9412888,9412888,both_failed,9412888
9,93,550,550,duplicate_id,550
